## Conditional Workflow 

In [6]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
import os
import operator
from dotenv import load_dotenv
from langchain_groq import ChatGroq

In [7]:
load_dotenv()

model = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.1-8b-instant",
    temperature=0.7
)

In [8]:
class SentimentScheme(BaseModel):
    sentiment:Literal["positive", "negative"] = Field(description="The sentiment of the review")
    

In [9]:
class DiagnosisScheme(BaseModel):
    issue_type:Literal["UX","Performance","Bug","Support","Other"] = Field(description="The category of issue mentioned in the review")
    tone:Literal["angry","frustrated","disappointed","calm"]= Field(description="The tone expressed by the user in the review")
    urgency:Literal["low","medium","high"]=Field(description="The urgency level of the issue mentioned in the review")

In [10]:
structured_model=model.with_structured_output(SentimentScheme)
structured_model1=model.with_structured_output(DiagnosisScheme)

In [11]:
class ReviewState(TypedDict):
    review:str
    sentiment:Literal["positive", "negative"]
    diagnosis:dict
    response:str


In [12]:
def find_sentiment(state: ReviewState):

    prompt = f'For the following review find out the sentiment \n {state["review"]}'
    sentiment = structured_model.invoke(prompt).sentiment

    return {'sentiment': sentiment}  #it will return either positive or negative

def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:

    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'
    
def positive_response(state: ReviewState):

    prompt = f"""Write a warm thank-you message in response to this review:
    \n\n\"{state['review']}\"\n
Also, kindly ask the user to leave feedback on our website."""
    
    response = model.invoke(prompt).content

    return {'response': response}

def run_diagnosis(state: ReviewState):   # used when we get the negative sentiment

    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n"
    "Return issue_type, tone, and urgency.
"""
    response = structured_model1.invoke(prompt)

    return {'diagnosis': response.model_dump()}

def negative_response(state: ReviewState):

    diagnosis = state['diagnosis']

    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response = model.invoke(prompt).content

    return {'response': response}
    

In [15]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')

graph.add_conditional_edges('find_sentiment', check_sentiment)

graph.add_edge('positive_response', END)

graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

In [19]:
# compile
workflow = graph.compile()

# execute
initial_state = {'review':"The product is slow and the support team is unresponsive. I'm very frustrated with this experience."}
final_state = workflow.invoke(initial_state)
print(final_state["response"])

**High Priority Issue Resolved**

Dear User,

I can sense the frustration and urgency behind your Performance issue report, and I want to assure you that I'm here to help you resolve it as soon as possible. I completely understand how critical performance issues can be, and I'm committed to getting you up and running smoothly as quickly as I can.

After reviewing your report, I have identified the root cause of the issue and have taken steps to address it. I'm pleased to inform you that the problem has been resolved, and your application should now be performing at its optimal level.

To confirm, please try the following:

1. Restart your application to ensure all changes have taken effect.
2. Check your application's performance metrics to verify that everything is running smoothly.

If you still encounter any issues or concerns, please don't hesitate to reach out to me directly. I'm here to support you 24/7 and will be more than happy to assist you further.

Thank you for your patien